# Step 0b — Cross-sectional confirmation (does SPY's finding generalize?)
SPY's step-0 gate passed on an omnibus criterion but flagged a real, well-powered
finding: the `bull_hi` cell showed WORSE 21d forward returns than `bear_hi`
(effect -1.08sd, n=300 vs 57), and `bear_lo` never occurred in 20 years of SPY data.
Before committing Phase 1 engineering effort to the 50-name architecture, this
notebook checks whether those two findings are an SPY/index-specific artifact or a
general single-name phenomenon, using the SAME fixed methodology (continuity-matched
HMM, no freeze) applied independently across a diversified liquid basket.

**Scope note:** this is a diagnostic, not the production per-name tilt layer. The
plan's two-layer design (market regime x name tilt) is unchanged -- fitting an
independent HMM per name here is purely to check whether the underlying price
behavior generalizes, not a proposal to run 50 independent HMMs in production.
**Designed to run in Google Colab** — Runtime > Run all.

In [ ]:
import importlib.util, subprocess, sys
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "yfinance", "hmmlearn"], check=True)
print("colab:", IN_COLAB)

In [ ]:
import warnings, json
from pathlib import Path
from itertools import permutations
import numpy as np
import pandas as pd
from scipy import stats as sstats
from scipy.stats import norm
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(11)

In [ ]:
OUTPUT_DIR = Path("/content/output") if IN_COLAB else Path("../../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_MODE = "auto"
START = "2005-01-01"
N_STATES = 3
REFIT_EVERY = 63
MIN_TRAIN = 756
FWD_HORIZONS = [5, 21]
COMMIT_P, COMMIT_DAYS, MIN_DWELL = 0.70, 2, 3
VOL_PCT_CENTER = 0.60          # per plan section 3.5: >60th pct realized-vol = high vol
COND_WINDOW, MIN_COND_OBS = 252, 40  # v2: direction-conditional vol percentile window/floor
                            # (trailing same-direction-label OBSERVATIONS, not calendar days) --
                            # see step0's v5 rationale: leverage effect means bear days clear an
                            # UNCONDITIONAL vol bar almost automatically (routine for that regime)
                            # while bull days crossing it are genuine anomalies, confounding the
                            # bull_hi-vs-bear_hi comparison. Conditioning removes that asymmetry.
# Diversified liquid basket: mega-cap tech, financials, healthcare, staples, energy,
# industrials, discretionary -- deliberately NOT all one sector, to avoid a sector-driven
# false confirmation/rejection.
TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "JPM", "BAC",
           "XOM", "CVX", "UNH", "JNJ", "V", "MA", "PG", "KO",
           "HD", "DIS", "CSCO", "WMT"]
print("universe:", len(TICKERS), "names")

In [ ]:
def try_live(tickers):
    import yfinance as yf
    px = yf.download(tickers, start=START, auto_adjust=True, progress=False)["Close"]
    return px

def make_synthetic_universe(tickers, n=2800):
    # Independent synthetic price series per name, same generator family as step0 but
    # with per-name seed variation and no shared vol-regime coupling -- structural test
    # of the LOOP/AGGREGATION code only, not evidence about real cross-sectional behavior.
    dates = pd.bdate_range("2015-01-01", periods=n)
    out = {}
    for i, tkr in enumerate(tickers):
        rng = np.random.RandomState(100 + i)
        mu = {0: -0.0010, 1: 0.0001, 2: 0.0007}
        sg = {0: 0.018, 1: 0.010, 2: 0.008}
        d_state = 2
        rets = []
        for t in range(n):
            if d_state == 2 and rng.rand() < 0.02: d_state = 1
            elif d_state == 1 and rng.rand() < 0.04: d_state = 0 if rng.rand() < 0.5 else 2
            elif d_state == 0 and rng.rand() < 0.035: d_state = 1
            vol_mult = 1.0 if rng.rand() > 0.15 else 2.2   # occasional independent vol spike
            rets.append(rng.normal(mu[d_state], sg[d_state] * vol_mult))
        out[tkr] = 100 * np.exp(np.cumsum(rets))
    return pd.DataFrame(out, index=dates)

MODE = DATA_MODE
if DATA_MODE == "auto":
    try:
        px = try_live(TICKERS); MODE = "live"
    except Exception as e:
        print(f"[!] live pull failed ({type(e).__name__}: {str(e)[:90]}) -> synthetic fallback")
        px = make_synthetic_universe(TICKERS); MODE = "synthetic"
else:
    px = make_synthetic_universe(TICKERS)
banner = "LIVE DATA — results are evidence" if MODE == "live" else \
         "SYNTHETIC DATA — STRUCTURAL RUN ONLY, ZERO EVIDENCE VALUE"
print("=" * 70 + f"\n  {banner}\n" + "=" * 70)
print(px.shape, px.index.min().date(), "->", px.index.max().date())

In [ ]:
# Reuse the fixed (v4) methodology from step0: continuity-matched HMM direction engine,
# no separation-based freeze, simple realized-vol-percentile vol layer (per plan 3.5 --
# no VIX-complex/curve inputs here since those are market-wide, not name-specific).
from hmmlearn.hmm import GaussianHMM

def fit_hmm(r, prev_means_ordered=None):
    m = GaussianHMM(n_components=N_STATES, covariance_type="diag",
                    n_iter=200, random_state=7, min_covar=1e-6)
    m.fit(r.reshape(-1, 1))
    raw_means = m.means_.ravel()
    if prev_means_ordered is None:
        order = np.argsort(raw_means)
    else:
        best_perm, best_cost = None, np.inf
        for perm in permutations(range(N_STATES)):
            cost = sum(abs(raw_means[perm[i]] - prev_means_ordered[i]) for i in range(N_STATES))
            if cost < best_cost:
                best_cost, best_perm = cost, perm
        order = np.array(best_perm)
    return m, order

def forward_filter(r, m, order):
    means = m.means_.ravel()[order]
    stds = np.sqrt(np.array([m.covars_[i].ravel()[0] for i in range(N_STATES)]))[order]
    A = m.transmat_[np.ix_(order, order)]
    pi = m.startprob_[order]
    alphas = np.zeros((len(r), N_STATES))
    lik = norm.pdf(r[:, None], means[None, :], stds[None, :]) + 1e-300
    a = pi * lik[0]; a /= a.sum(); alphas[0] = a
    for t in range(1, len(r)):
        a = (A.T @ a) * lik[t]; a /= a.sum(); alphas[t] = a
    return alphas, A

def conditional_pct(raw, dir_label, window=COND_WINDOW, min_obs=MIN_COND_OBS):
    out = pd.Series(np.nan, index=raw.index)
    for g in dir_label.dropna().unique():
        idx = dir_label[dir_label == g].index
        sub = raw.reindex(idx)
        out.loc[idx] = sub.rolling(window, min_periods=min_obs).rank(pct=True)
    return out

def analyze_ticker(prices):
    ret = np.log(prices).diff().dropna()
    rv20 = ret.rolling(20).std() * np.sqrt(252)
    rv_pct = rv20.rolling(252).rank(pct=True)
    feat = pd.DataFrame({"ret": ret, "rv20": rv20, "rv_pct": rv_pct}).dropna()
    if len(feat) < MIN_TRAIN + REFIT_EVERY:
        return None
    r = feat["ret"].values
    n = len(r)
    post = np.full((n, N_STATES), np.nan)
    prev_means_ordered = None
    for t in range(MIN_TRAIN, n, REFIT_EVERY):
        m, order = fit_hmm(r[:t], prev_means_ordered)
        prev_means_ordered = m.means_.ravel()[order]
        seg_end = min(t + REFIT_EVERY, n)
        alphas, _ = forward_filter(r[:seg_end], m, order)
        post[t:seg_end] = alphas[t:seg_end]
    dirpost = pd.DataFrame(post, index=feat.index, columns=["p_bear", "p_neut", "p_bull"]).dropna()
    # v2: direction-conditional vol percentile (see COND_WINDOW rationale above), falls
    # back to the unconditional percentile when a direction state has too little history.
    dir_label = dirpost.idxmax(axis=1).map({"p_bear": "bear", "p_neut": "neut", "p_bull": "bull"})
    rv_pct_cond = conditional_pct(feat["rv20"].reindex(dirpost.index), dir_label)
    rv_pct_final = rv_pct_cond.fillna(feat["rv_pct"].reindex(dirpost.index))
    p_high = 1 / (1 + np.exp(-8 * (rv_pct_final - VOL_PCT_CENTER)))
    cells = ["bear_hi", "bear_lo", "neut_hi", "neut_lo", "bull_hi", "bull_lo"]
    cp = pd.DataFrame(index=dirpost.index, columns=cells, dtype=float)
    for i, d in enumerate(["p_bear", "p_neut", "p_bull"]):
        cp[cells[2 * i]] = dirpost[d] * p_high
        cp[cells[2 * i + 1]] = dirpost[d] * (1 - p_high)
    raw_cell = cp.idxmax(axis=1)
    committed, cur, streak, dwell = [], None, 0, 0
    for t in range(len(cp)):
        top, ptop = raw_cell.iloc[t], cp.iloc[t].max()
        if cur is None: cur = top
        else:
            streak = streak + 1 if (top != cur and ptop >= COMMIT_P) else 0
            if streak >= COMMIT_DAYS and dwell >= MIN_DWELL:
                cur, streak, dwell = top, 0, 0
        dwell += 1
        committed.append(cur)
    committed = pd.Series(committed, index=cp.index)
    out = {"n_obs": int(len(committed)), "cell_counts": committed.value_counts().to_dict()}
    for h in FWD_HORIZONS:
        fr = feat["ret"].reindex(committed.index).rolling(h).sum().shift(-h)
        g = pd.DataFrame({"cell": committed, "fr": fr}).dropna()
        for vb in ["hi", "lo"]:
            bull = g.loc[g.cell == f"bull_{vb}", "fr"]; bear = g.loc[g.cell == f"bear_{vb}", "fr"]
            key = f"{h}_{vb}"
            if len(bull) > 30 and len(bear) > 30:
                ks = sstats.ks_2samp(bull, bear)
                out[f"effect_{key}"] = float((bull.mean() - bear.mean()) / g["fr"].std())
                out[f"ks_p_{key}"] = float(ks.pvalue)
                out[f"n_bull_{key}"] = int(len(bull)); out[f"n_bear_{key}"] = int(len(bear))
            else:
                out[f"effect_{key}"] = None; out[f"ks_p_{key}"] = None
                out[f"n_bull_{key}"] = int(len(bull)); out[f"n_bear_{key}"] = int(len(bear))
    return out

In [ ]:
results = {}
for tkr in TICKERS:
    try:
        s = px[tkr].dropna() if isinstance(px, pd.DataFrame) else px.dropna()
        r = analyze_ticker(s)
        results[tkr] = r
        status = "ok" if r else "insufficient history"
        print(f"{tkr:6s}  {status}")
    except Exception as e:
        results[tkr] = {"error": str(e)[:120]}
        print(f"{tkr:6s}  FAILED: {str(e)[:100]}")
n_ok = sum(1 for v in results.values() if v and "error" not in v)
print(f"\n{n_ok}/{len(TICKERS)} names analyzed successfully")

In [ ]:
# Aggregate: does the SPY pattern (bull_hi reversal, bear_lo rarity) generalize?
rows = []
for tkr, r in results.items():
    if not r or "error" in r: continue
    row = {"ticker": tkr, "n_obs": r["n_obs"]}
    for h in FWD_HORIZONS:
        for vb in ["hi", "lo"]:
            row[f"effect_{h}_{vb}"] = r.get(f"effect_{h}_{vb}")
            row[f"n_bear_{h}_{vb}"] = r.get(f"n_bear_{h}_{vb}")
    row["bear_lo_ever_occurs"] = (r["cell_counts"].get("bear_lo", 0) > 0)
    rows.append(row)
agg = pd.DataFrame(rows).set_index("ticker")
print(agg.round(3).to_string())

n_bear_lo_present = int(agg["bear_lo_ever_occurs"].sum())
print(f"\nbear_lo occurs in {n_bear_lo_present}/{len(agg)} names "
      f"(SPY: 0/1 -- {'confirms SPY finding' if n_bear_lo_present <= len(agg)*0.2 else 'SPY finding does NOT generalize'})")

e21hi = agg["effect_21_hi"].dropna()
if len(e21hi) >= 3:
    neg_share = (e21hi < 0).mean()
    print(f"21d bull_hi-vs-bear_hi effect: {len(e21hi)} names with sufficient data, "
          f"median={e21hi.median():.2f}sd, {neg_share:.0%} negative (SPY was -1.08sd)")
    print("-> SPY's reversal looks GENERAL across the universe" if neg_share > 0.5 else
          "-> SPY's reversal looks INDEX-SPECIFIC, not present in most single names")
else:
    print("insufficient names with populated bull_hi/bear_hi cells to judge generality")

In [ ]:
# Visual: distribution of the 21d bull_hi-vs-bear_hi effect across the universe, SPY overlay
fig, ax = plt.subplots(figsize=(9, 4))
vals = agg["effect_21_hi"].dropna()
if len(vals) > 0:
    ax.hist(vals, bins=max(5, len(vals)//3), color="#4c78a8", edgecolor="white")
    ax.axvline(-1.08, color="firebrick", lw=2, label="SPY (-1.08sd)")
    ax.axvline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel("21d bull_hi vs bear_hi effect size (sd)"); ax.set_title(f"Cross-sectional check [{MODE}]")
    ax.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "step0b_crosssectional.png", dpi=110); plt.show()
print("chart saved")

In [ ]:
summary = {
    "mode": MODE,
    "n_names_analyzed": n_ok,
    "bear_lo_occurs_in_n_names": n_bear_lo_present,
    "bear_lo_total_names": int(len(agg)),
    "per_ticker": {k: v for k, v in results.items() if v and "error" not in v},
    "errors": {k: v for k, v in results.items() if v and "error" in v},
}
with open(OUTPUT_DIR / "step0b_verdict.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print("verdict written ->", OUTPUT_DIR / "step0b_verdict.json")
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(OUTPUT_DIR / "step0b_verdict.json"))
    except Exception as e:
        print("(auto-download skipped:", e, "- grab it from the Files pane)")